# SE201 Week 5 · Base and instruct variants of the same model

Two checkpoints of **the same model**. They have the same architecture, same parameter count, and same tokeniser. 

The base model is the product of pre-training.

The instruct model is the product of supervised fine-tuning

Four things to look for:

1. The base model **continues your text**; the instruct model **answers your question**.
2. The instruct model expects a **chat template** — special tokens that mark whose turn it is.
3. A few examples make the base model answer too. The knowledge was always there, the
   *behaviour* is what stage 3 added.
4. The instruct model **stops**; the base model keeps writing until it runs out of budget.

**Before you start:** run the first two cells. They download about 2 GB in total, which
takes a few minutes on a good connection. Everything after that runs on a CPU, a few
seconds per generation.

In [1]:
# Run once. If you are on Colab, restart the runtime afterwards only if it asks you to.
%pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# The pair. Both are ungated: no Hugging Face token needed. Both have the same underlying architecture (same number of blocks etc.), but the INSTRUCT has fine-tuned weights
BASE     = 'Qwen/Qwen2.5-0.5B'
INSTRUCT = 'Qwen/Qwen2.5-0.5B-Instruct' #trained by alibaba with stage 4 (RLHF etc.)

# Lower-bandwidth alternative (~270 MB each, weaker text on both sides):
# BASE, INSTRUCT = 'HuggingFaceTB/SmolLM2-135M', 'HuggingFaceTB/SmolLM2-135M-Instruct'

if torch.cuda.is_available():
    DEVICE, DTYPE = 'cuda', torch.bfloat16
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE, DTYPE = 'mps', torch.float32
else:
    DEVICE, DTYPE = 'cpu', torch.float32
print('device:', DEVICE, '| dtype:', DTYPE)

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=DTYPE).to(DEVICE)
    model.eval()
    return tok, model

tok_b, base = load(BASE)
tok_i, inst = load(INSTRUCT)
print('loaded')


device: mps | dtype: torch.float32


ValueError: Tokenizer class Qwen2Tokenizer does not exist or is not currently imported.

## They are the same network

Before comparing behaviour, confirm that nothing structural differs. Every field here
is one you can read off a model card (Week 5, slide 6).


In [4]:
def describe(name, model, tok):
    c = model.config
    return {
        'model': name.split('/')[-1],
        'blocks': c.num_hidden_layers,
        'd_model': c.hidden_size,
        'heads': c.num_attention_heads,
        'kv_heads': getattr(c, 'num_key_value_heads', c.num_attention_heads),
        'vocab': c.vocab_size,
        'parameters': f'{sum(p.numel() for p in model.parameters())/1e6:.1f} M',
    }

rows = [describe(BASE, base, tok_b), describe(INSTRUCT, inst, tok_i)]
cols = list(rows[0])
width = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
print('  '.join(c.ljust(width[c]) for c in cols))
for r in rows:
    print('  '.join(str(r[c]).ljust(width[c]) for c in cols))


NameError: name 'base' is not defined

---
## Part 1 — The same prompt, no template

Plain text in, plain continuation out. No chat formatting, no system prompt, greedy
decoding so the result is reproducible.


In [ ]:
@torch.no_grad()
def complete(tok, model, text, max_new_tokens=200, temperature=0.0, seed=None):
    """Continue `text`. temperature=0 means greedy (Week 5)."""
    if seed is not None:
        torch.manual_seed(seed)
    ids = tok(text, return_tensors='pt').to(DEVICE)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature and temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kwargs.update(do_sample=False)
    out = model.generate(**ids, **kwargs)
    new = out[0][ids['input_ids'].shape[1]:]
    return tok.decode(new, skip_special_tokens=False)

def show(label, text):
    print('=' * 78)
    print(label)
    print('-' * 78)
    print(text.strip() if text.strip() else '(nothing)')
    print()

PROMPT = "Rewrite the following sentence in the passive voice: The compiler optimises the loop."
show('BASE      ' + BASE,     complete(tok_b, base, PROMPT))
show('INSTRUCT  ' + INSTRUCT, complete(tok_i, inst, PROMPT))


**What did each one do with the question?**

The base model was trained to predict the next token in a document.

On the internet a question is very often followed by more questions, such as a quiz, a FAQ, or an
exercise sheet, and continuing with more questions is a *good* prediction. It is not
being unhelpful. Helpfulness was never in its objective.

---
## Part 2 — The chat template

The instruct model was trained on conversations marked up with special tokens: one to
open a turn, one to close it, and a role name in between. `apply_chat_template` builds
that markup for you. Print it before using it.


In [ ]:
messages = [{'role': 'user', 'content': PROMPT}]
templated = tok_i.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(repr(templated))
print()
print('instruct special tokens:', tok_i.all_special_tokens)
print('base     special tokens:', tok_b.all_special_tokens)
print()
print('base repo ships a chat template:', getattr(tok_b, 'chat_template', None) is not None)


Note the last line. A base repository often *ships* a chat template anyway, because the
file is inherited from the same tokeniser configuration. 

A template does not tell you if the model was ever **trained** to follow it.

The next cell shows what happens when you give the template to a model that was never trained on it.

In [ ]:
show('INSTRUCT + template  (what you should do)', complete(tok_i, inst, templated))
show('BASE + template      (tokens it never learned a role for)', complete(tok_b, base, templated))
show('INSTRUCT, raw prompt (no template)',        complete(tok_i, inst, PROMPT))

### Question:
Do you see a potential security problem with this style of controlling an LLM in this manner?

---
## Part 3 — The capability was already there

If the base model cannot answer a question, is that because it does not know how?
Give it a few worked examples first — no fine-tuning, no weights changed, just a prefix
that makes an answer the likely continuation.

In [ ]:
#example of fewshot learning (give examples in the prompt and let the model complete the next example)
FEWSHOT = (
    "Rewrite each sentence in the passive voice.\n\n"
    "Active: The garbage collector reclaims the memory.\n"
    "Passive: The memory is reclaimed by the garbage collector.\n\n"
    "Active: The linker resolves the symbol.\n"
    "Passive: The symbol is resolved by the linker.\n\n"
    "Active: The compiler optimises the loop.\n"
    "Passive: "
)

answer = complete(tok_b, base, FEWSHOT, max_new_tokens=200)
show('BASE, three examples first', answer.split('\n')[0])

# For contrast, the same model with no examples:
show('BASE, no examples', complete(tok_b, base, PROMPT, max_new_tokens=200))


The base model had the capability the whole time. 

The supervised finetuning adds *behaviour*, not knowledge.

It is why fine-tuning is the wrong tool to add knowledge when a model does not know something, and
retrieval is the right one.

**Try it:** find a question where the few-shot prefix does *not* rescue the base model.
What does that tell you about which stage is responsible?


---
## Part 4 — Stopping

An instruct model has been trained to emit an end-of-turn token to halt generation.

A base model has no notion of a turn and it stops when it runs out of budget.

In [ ]:
PROMPT = "List three Python built-in functions. Answer with only the three names, comma-separated."
templated = tok_i.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

for label, tok, model, text in [
    ('BASE',     tok_b, base, PROMPT),
    ('INSTRUCT', tok_i, inst, templated),
]:
    ids = tok(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=200, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    produced = out[0][ids['input_ids'].shape[1]:]
    stopped_early = len(produced) < 200
    print(f'{label:9s} eos token: {tok.eos_token!r} (id {tok.eos_token_id})')
    print(f'{label:9s} produced {len(produced)} of 200 tokens '
          f'-> {"stopped on its own" if stopped_early else "hit the limit"}')
    print(f'{label:9s} last token: {tok.decode(produced[-1:])!r}')
    print()

In [ ]:
show('BASE      ' + BASE,     complete(tok_b, base, PROMPT)) # did not know when to stop listing
show('INSTRUCT  ' + INSTRUCT, complete(tok_i, inst, PROMPT)) # followed the instruction to stop after three names

This is the difference between a well-behaved API response and one that runs on until it is cut off.

---
## Part 5 — The distribution

One forward pass produces one distribution over the vocabulary which yields one token. Here are the five most likely next tokens after the same prefix, for each model.

In [ ]:
@torch.no_grad()
def top5(tok, model, text):
    ids = tok(text, return_tensors='pt').to(DEVICE)
    logits = model(**ids).logits[0, -1]
    probs = torch.softmax(logits.float(), dim=-1)
    p, idx = torch.topk(probs, 5)
    return [(tok.decode([int(i)]), round(v.item(), 4)) for v, i in zip(p, idx)]

PREFIX = 'The capital of France is'
print('BASE     ', top5(tok_b, base, PREFIX))
print('INSTRUCT ', top5(tok_i, inst, PREFIX))


And the sampler is a choice made by the code around the model, not a property of the
model (Week 5, slide 12). Same weights, same prompt, three draws at T = 0.8:


In [ ]:
for run in range(3):
    print(f'--- T=0.8, run {run + 1}')
    print(complete(tok_i, inst, templated, max_new_tokens=40, temperature=0.8, seed=run).strip())
    print()

print('--- T=0 (greedy), twice')
for run in range(2):
    print(complete(tok_i, inst, templated, max_new_tokens=40).strip())


---
## Your turn

1. Ask both models something that requires following an instruction rather than
   continuing text — *"Summarise this in one sentence: ..."*, *"Rewrite this as a
   commit message: ..."*. Does the few-shot trick rescue the base model there too?

2. Build a two-turn conversation with `apply_chat_template` and check what markup the
   second turn produces.